# 🎙️ Speech Emotion Recognition Using Machine Learning
---
This notebook builds a complete Speech Emotion Recognition (SER) system step by step.

The system listens to audio recordings and detects the emotion of the speaker —
such as happy, sad, angry, neutral, fearful, and more.

We follow a clear pipeline:
1. Load and explore the datasets
2. Clean and prepare the audio files
3. Extract features from each audio file
4. Train and compare four models
5. Evaluate and present the final results

All four datasets are free and available on Kaggle.
All experiments run on Kaggle free GPU — no local setup needed.

## Section 1 — Import Libraries
---
In this section we import all the Python libraries we need for the entire project.

- **librosa** — reads audio files and extracts features like MFCC
- **numpy and pandas** — handle data and organise it into tables
- **scikit-learn** — used for the SVM baseline model and data scaling
- **PyTorch** — used to build CNN, CNN-LSTM and Attention deep learning models
- **matplotlib and seaborn** — used to plot graphs and confusion matrices
- **os and glob** — used to navigate folders and find audio files

In [ ]:
# ─────────────────────────────────────────────
# SECTION 1 — IMPORT LIBRARIES
# ─────────────────────────────────────────────

# Audio processing
import librosa
import librosa.display

# Data handling
import numpy as np
import pandas as pd

# File and folder navigation
import os
import glob

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report,
                             confusion_matrix)

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Progress bar
from tqdm import tqdm

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("All libraries imported successfully.")

## Section 2 — Load Datasets
---
We use four publicly available speech emotion datasets. Each dataset contains
audio recordings of actors speaking with different emotions.

| Dataset | Speakers | Emotions | Files |
|---------|----------|----------|-------|
| RAVDESS | 24 professional actors | 8 emotions | ~1,440 |
| CREMA-D | 91 actors | 6 emotions | 7,442 |
| TESS | 2 female speakers | 7 emotions | 2,800 |
| SAVEE | 4 male speakers | 7 emotions | 480 |

All datasets are already added to this notebook via Kaggle.
We load all audio file paths and their emotion labels into one combined dataframe.

The emotion labels across all datasets are mapped to these common categories:
angry, disgust, fear, happy, neutral, sad, surprised, calm.

In [ ]:
# ── Label extractor functions ──

import os
import glob
import pandas as pd

def get_ravdess_label(filepath):
    filename = os.path.basename(filepath)
    parts = filename.split('-')
    if len(parts) < 3:
        return None
    try:
        emotion_code = int(parts[2])
    except ValueError:
        return None
    emotion_map = {
        1: 'neutral', 2: 'calm',     3: 'happy', 4: 'sad',
        5: 'angry',   6: 'fearful',  7: 'disgust', 8: 'surprised'
    }
    return emotion_map.get(emotion_code, None)

def get_cremad_label(filepath):
    filename = os.path.basename(filepath)
    parts = filename.split('_')
    if len(parts) < 3:
        return None
    emotion_code = parts[2]
    emotion_map = {
        'ANG': 'angry',   'DIS': 'disgust', 'FEA': 'fearful',
        'HAP': 'happy',   'NEU': 'neutral', 'SAD': 'sad'
    }
    return emotion_map.get(emotion_code, None)

def get_tess_label(filepath):
    folder = os.path.basename(os.path.dirname(filepath)).lower()
    emotion_map = {
        'angry':   'angry',   'disgust': 'disgust',
        'fear':    'fearful', 'happy':   'happy',
        'neutral': 'neutral', 'sad':     'sad',
        'ps':      'surprised'
    }
    for key in emotion_map:
        if key in folder:
            return emotion_map[key]
    return None

def get_savee_label(filepath):
    filename = os.path.basename(filepath).lower()
    parts = filename.split('_')
    if len(parts) < 2:
        return None
    code = parts[1][:2].strip()
    emotion_map = {
        'a':  'angry',   'd':  'disgust', 'f':  'fearful',
        'h':  'happy',   'n':  'neutral', 'sa': 'sad',
        'su': 'surprised'
    }
    return emotion_map.get(code, emotion_map.get(code[0], None))

print("Label functions defined successfully.")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2 — LOAD DATASETS (FIXED PATHS)
# ─────────────────────────────────────────────

def load_all_datasets():
    data = []

    # ── RAVDESS ──
    ravdess_files = glob.glob(
        '/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio/**/*.wav',
        recursive=True
    )
    for f in ravdess_files:
        label = get_ravdess_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'RAVDESS'})

    # ── CREMA-D ──
    # CREMA-D is inside the dmitrybabko combined pack
    cremad_files = glob.glob(
        '/kaggle/input/datasets/dmitrybabko/speech-emotion-recognition-en/Crema/**/*.wav',
        recursive=True
    )
    for f in cremad_files:
        label = get_cremad_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'CREMAD'})

    # ── TESS ──
    tess_files = glob.glob(
        '/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess/**/*.wav',
        recursive=True
    )
    for f in tess_files:
        label = get_tess_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'TESS'})

    # ── SAVEE ──
    savee_files = glob.glob(
        '/kaggle/input/datasets/dmitrybabko/speech-emotion-recognition-en/Savee/**/*.wav',
        recursive=True
    )
    for f in savee_files:
        label = get_savee_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'SAVEE'})

    return pd.DataFrame(data)


# Load everything
df = load_all_datasets()

print(f"Total audio files loaded: {len(df)}")
print(f"\nFiles per dataset:")
print(df['source'].value_counts())
print(f"\nEmotion distribution:")
print(df['emotion'].value_counts())

In [ ]:
# ── 2.3 Visualise emotion distribution ──

plt.figure(figsize=(12, 4))

# Emotion count
plt.subplot(1, 2, 1)
df['emotion'].value_counts().plot(kind='bar', color='#2e7d32')
plt.title('Emotion Distribution (All Datasets)')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)

# Dataset count
plt.subplot(1, 2, 2)
df['source'].value_counts().plot(kind='bar', color='#1b5e20')
plt.title('Files per Dataset')
plt.xlabel('Dataset')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()